<a href="https://colab.research.google.com/github/suphanatchanlek30/Super-AI-Engineer-Season-6-Thai-Call-Center-ASR/blob/main/Thai_Call_Center_ASR_600367_%E0%B8%A8%E0%B8%B8%E0%B8%A0%E0%B8%93%E0%B8%B1%E0%B8%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thai Call Center ASR - Kaggle/Colab inference

Baseline for **Super AI Engineer SS6 Individual Test: Thai Call Center ASR**.

The notebook:

- uses `typhoon-ai/typhoon-whisper-large-v3` through Hugging Face Transformers;
- auto-detects `sample_submission.csv` and the WAV directory;
- resumes from `checkpoint_predictions.csv`;
- saves a checkpoint after every 100 newly inferred files;
- removes ASR control tokens and punctuation without deleting Thai filler words;
- can infer one original recording and copy its transcript to `_phone`, `_noise`,
  `_fast`, `_slow`, and `_pitch` variants;
- validates row count, columns, missing values, and exact sample file order before
  writing `submission.csv`.

For Kaggle, enable **GPU** and **Internet** (unless the model is attached as a
Kaggle dataset). Large-v3 prioritizes accuracy. If the GPU/runtime is too limited,
change `MODEL_ID` to `typhoon-ai/typhoon-whisper-turbo`.


In [ ]:
# Install only the packages not reliably available in every Kaggle/Colab image.
# PyTorch is intentionally not reinstalled because the hosted runtime supplies
# a CUDA-compatible build.
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.48,<5",
    "accelerate>=1.0",
    "safetensors>=0.4",
    "librosa>=0.10.2",
    "soundfile>=0.12",
])
print("Dependencies installed.")


In [ ]:
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

DATA_DIR = Path(os.getenv("ASR_DATA_DIR", "./data")).resolve()
COMPETITION = os.getenv("ASR_COMPETITION", "individual-test-thai-call-center-asr")


def has_dataset(root: Path) -> bool:
    return any(root.rglob("sample_submission.csv")) and any(root.rglob("*.wav"))


def find_kaggle_json() -> Path:
    candidates = [
        Path(os.getenv("KAGGLE_JSON", "kaggle.json")).expanduser(),
        Path.home() / ".kaggle" / "kaggle.json",
    ]
    for path in candidates:
        if path.is_file():
            try:
                payload = json.loads(path.read_text(encoding="utf-8-sig"))
            except Exception:
                continue
            if payload.get("username") and payload.get("key"):
                return path.resolve()
    raise FileNotFoundError(
        "kaggle.json not found or invalid. Put it beside notebook or in ~/.kaggle/kaggle.json"
    )


def safe_extract(zip_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            target.relative_to(destination)
        archive.extractall(destination)


def extract_nested(root: Path) -> None:
    seen = set()
    while True:
        zips = [p for p in root.rglob("*.zip") if p.resolve() not in seen]
        if not zips:
            break
        for path in zips:
            safe_extract(path, path.parent)
            seen.add(path.resolve())


DATA_DIR.mkdir(parents=True, exist_ok=True)
if not has_dataset(DATA_DIR):
    kaggle_json = find_kaggle_json()
    os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_json.parent)

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"])
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    api.competition_download_files(COMPETITION, path=str(DATA_DIR), quiet=False)

    zip_candidates = list(DATA_DIR.glob("*.zip"))
    if not zip_candidates:
        raise FileNotFoundError(f"No zip downloaded in {DATA_DIR}")

    safe_extract(zip_candidates[0], DATA_DIR)
    extract_nested(DATA_DIR)

if not has_dataset(DATA_DIR):
    raise RuntimeError(f"Dataset not ready under {DATA_DIR}")

samples = list(DATA_DIR.rglob("sample_submission.csv"))
if not samples:
    raise FileNotFoundError("sample_submission.csv not found after extraction")

sample_csv = samples[0].resolve()
audio_candidates = [sample_csv.parent / "audio", sample_csv.parent / "audio_final", sample_csv.parent]
audio_dir = next((p for p in audio_candidates if p.is_dir() and any(p.rglob("*.wav"))), None)
if audio_dir is None:
    audio_dir = next((p for p in DATA_DIR.rglob("*") if p.is_dir() and any(p.rglob("*.wav"))), None)
if audio_dir is None:
    raise FileNotFoundError("No directory with WAV files found")

os.environ["ASR_SAMPLE_CSV"] = str(sample_csv)
os.environ["ASR_AUDIO_DIR"] = str(audio_dir.resolve())

print("Data ready")
print("ASR_SAMPLE_CSV:", os.environ["ASR_SAMPLE_CSV"])
print("ASR_AUDIO_DIR:", os.environ["ASR_AUDIO_DIR"])


## Optional: Download Kaggle Data (Run Once)

If the target machine does not already have the dataset, run the next cell first.

Requirements:
- `kaggle.json` available in the notebook folder or `~/.kaggle/kaggle.json`
- competition rules already accepted on Kaggle
- internet access

In [ ]:
import gc
import os
import re
import time
import unicodedata
from collections import defaultdict
from pathlib import Path

import pandas as pd
import librosa
import numpy as np
import soundfile as sf
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# ------------------------------- Configuration -------------------------------
MODEL_ID = os.getenv(
    "ASR_MODEL_ID",
    "typhoon-ai/typhoon-whisper-large-v3",
)
# Faster/lighter fallback:
# MODEL_ID = "typhoon-ai/typhoon-whisper-turbo"

# Leave these as None for automatic Kaggle/Colab/local discovery.
SAMPLE_CSV = os.getenv("ASR_SAMPLE_CSV") or None
AUDIO_DIR = os.getenv("ASR_AUDIO_DIR") or None

OUTPUT_DIR = Path(
    os.getenv("ASR_OUTPUT_DIR")
    or (
        "/kaggle/working"
        if Path("/kaggle/working").exists()
        else "/content"
        if Path("/content").exists()
        else Path.cwd()
    )
)
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint_predictions.csv"
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"

USE_ORIGINAL_FOR_VARIANTS = (
    os.getenv("ASR_USE_ORIGINAL_FOR_VARIANTS", "1").lower()
    not in {"0", "false", "no"}
)
CHECKPOINT_EVERY = int(os.getenv("ASR_CHECKPOINT_EVERY", "100"))
BATCH_SIZE = int(os.getenv("ASR_BATCH_SIZE", "4"))
CHUNK_LENGTH_S = int(os.getenv("ASR_CHUNK_LENGTH_S", "30"))
MIN_CHUNK_LENGTH_S = int(os.getenv("ASR_MIN_CHUNK_LENGTH_S", "5"))
STRIDE_LENGTH_S = (4, 2)
# Whisper reserves decoder positions for language/task start tokens.
# Keep this below max_target_positions=448 (Thai prompt currently uses 3).
MAX_NEW_TOKENS = int(os.getenv("ASR_MAX_NEW_TOKENS", "440"))

ALLOW_CPU_FALLBACK = (
    os.getenv("ASR_ALLOW_CPU_FALLBACK", "1").lower() not in {"0", "false", "no"}
)

# Set a small integer (for example 8) only for a smoke test. The final
# submission cell intentionally refuses to export incomplete predictions.
MAX_MODEL_FILES = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def is_cuda_oom(error):
    return isinstance(error, torch.cuda.OutOfMemoryError) or (
        "out of memory" in str(error).lower()
        and "cuda" in str(error).lower()
    )


In [ ]:
def directory_has_wav(path):
    path = Path(path)
    if not path.is_dir():
        return False
    try:
        next(path.rglob("*.wav"))
        return True
    except StopIteration:
        return False


def find_dataset(sample_csv=None, audio_dir=None):
    if sample_csv is not None:
        sample_path = Path(sample_csv)
        if not sample_path.exists():
            raise FileNotFoundError(f"SAMPLE_CSV does not exist: {sample_path}")
    else:
        search_roots = [
            Path("/kaggle/input"),
            Path("/content"),
            Path.cwd(),
        ]
        candidates = []
        seen = set()
        for root in search_roots:
            if not root.exists():
                continue
            for path in root.rglob("sample_submission.csv"):
                resolved = path.resolve()
                if resolved not in seen:
                    seen.add(resolved)
                    candidates.append(path)
        if not candidates:
            raise FileNotFoundError(
                "Could not find sample_submission.csv. Set SAMPLE_CSV explicitly."
            )

        # Prefer a sample whose parent contains an obvious WAV directory.
        def sample_score(path):
            parent = path.parent
            likely_dirs = [parent / "audio", parent / "audio_final", parent]
            return max(
                [3 - i for i, candidate in enumerate(likely_dirs)
                 if directory_has_wav(candidate)]
                or [0]
            )

        sample_path = max(candidates, key=sample_score)

    if audio_dir is not None:
        wav_dir = Path(audio_dir)
        if not directory_has_wav(wav_dir):
            raise FileNotFoundError(f"No WAV files found under AUDIO_DIR: {wav_dir}")
    else:
        parent = sample_path.parent
        likely_dirs = [parent / "audio", parent / "audio_final", parent]
        wav_dir = next(
            (candidate for candidate in likely_dirs if directory_has_wav(candidate)),
            None,
        )
        if wav_dir is None:
            raise FileNotFoundError(
                f"Could not find WAV files near {sample_path}. Set AUDIO_DIR explicitly."
            )

    return sample_path, wav_dir


sample_path, audio_dir = find_dataset(SAMPLE_CSV, AUDIO_DIR)
sample = pd.read_csv(sample_path)

expected_columns = ["file_name", "text"]
if sample.columns.tolist() != expected_columns:
    raise ValueError(
        f"Expected sample columns {expected_columns}, got {sample.columns.tolist()}"
    )
if sample["file_name"].isna().any():
    raise ValueError("sample_submission.csv contains null file_name values.")
if sample["file_name"].duplicated().any():
    duplicates = sample.loc[sample["file_name"].duplicated(), "file_name"].head().tolist()
    raise ValueError(f"sample_submission.csv contains duplicate file names: {duplicates}")

sample["file_name"] = sample["file_name"].astype(str)
sample_names = sample["file_name"].tolist()

audio_index = {}
duplicate_audio_names = []
for wav_path in audio_dir.rglob("*.wav"):
    if wav_path.name in audio_index:
        duplicate_audio_names.append(wav_path.name)
    audio_index[wav_path.name] = wav_path

if duplicate_audio_names:
    raise ValueError(
        "Duplicate WAV basenames found in nested directories: "
        f"{duplicate_audio_names[:5]}"
    )

missing_audio = [name for name in sample_names if name not in audio_index]
if missing_audio:
    raise FileNotFoundError(
        f"{len(missing_audio)} sample files are missing from {audio_dir}. "
        f"Examples: {missing_audio[:5]}"
    )

print("sample:", sample_path)
print("audio directory:", audio_dir)
print("sample rows:", len(sample))
print("indexed WAV files:", len(audio_index))
print("checkpoint:", CHECKPOINT_PATH)


In [ ]:
VARIANT_RE = re.compile(r"_(phone|noise|fast|slow|pitch)(?=\.wav$)", re.IGNORECASE)
WHISPER_TOKEN_RE = re.compile(r"<\|.*?\|>")
NOISE_TAG_RE = re.compile(
    r"\[(?:music|noise|silence|laughter|laugh|applause|inaudible)\]",
    re.IGNORECASE,
)


def original_file_name(file_name):
    '''Map foo_noise.wav (and other known variants) back to foo.wav.'''
    return VARIANT_RE.sub("", file_name)


def clean_thai_transcript(text):
    '''
    Remove model control tokens, annotation tags, punctuation, symbols, and
    unexpected scripts. Keep Thai (including ๆ/ฯ and Thai digits), ASCII
    letters/digits, and whitespace. Thai filler words are ordinary Thai text
    and are therefore preserved.
    '''
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = WHISPER_TOKEN_RE.sub("", text)
    text = NOISE_TAG_RE.sub("", text)
    text = "".join(
        char
        for char in text
        if (
            "\u0E00" <= char <= "\u0E7F"
            or "a" <= char.lower() <= "z"
            or "0" <= char <= "9"
            or char.isspace()
        )
        and unicodedata.category(char) not in {"Cc", "Cf"}
    )
    return re.sub(r"\s+", " ", text).strip()


def load_checkpoint(path, valid_names):
    predictions = {}
    path = Path(path)
    if not path.exists():
        return predictions

    checkpoint = pd.read_csv(path, keep_default_na=False)
    required = {"file_name", "text"}
    if not required.issubset(checkpoint.columns):
        raise ValueError(
            f"Checkpoint must contain {sorted(required)}; got {checkpoint.columns.tolist()}"
        )
    checkpoint = checkpoint.drop_duplicates("file_name", keep="last")
    valid_names = set(valid_names)
    for row in checkpoint.itertuples(index=False):
        name = str(row.file_name)
        if name in valid_names:
            predictions[name] = clean_thai_transcript(row.text)
    return predictions


def save_checkpoint(path, ordered_names, predictions):
    rows = [
        {"file_name": name, "text": predictions[name]}
        for name in ordered_names
        if name in predictions
    ]
    frame = pd.DataFrame(rows, columns=["file_name", "text"])
    path = Path(path)
    temp_path = path.with_name(path.name + ".tmp")
    frame.to_csv(temp_path, index=False, encoding="utf-8")
    os.replace(temp_path, path)


groups = defaultdict(list)
for name in sample_names:
    groups[original_file_name(name)].append(name)


def propagate_available_originals(predictions):
    replacements = 0
    if not USE_ORIGINAL_FOR_VARIANTS:
        return replacements
    for original, members in groups.items():
        if original in predictions:
            original_text = predictions[original]
            for member in members:
                if predictions.get(member) != original_text:
                    predictions[member] = original_text
                    replacements += 1
    return replacements


def build_inference_queue(predictions):
    if not USE_ORIGINAL_FOR_VARIANTS:
        return [name for name in sample_names if name not in predictions]

    queue = []
    for original, members in groups.items():
        # Prefer one original recording whenever its WAV is available, even
        # when only augmented variants appear in the submission rows.
        if original in audio_index:
            if any(member not in predictions for member in members):
                queue.append(original)
        else:
            queue.extend(name for name in members if name not in predictions)
    return queue


# Small deterministic checks before loading a multi-GB model.
assert original_file_name("RSP_001_audio_noise.wav") == "RSP_001_audio.wav"
assert original_file_name("RSP_001_audio.wav") == "RSP_001_audio.wav"
cleaned_example = clean_thai_transcript(
    "<|th|> เอ่อ... อืม, ได้ค่ะ! ครับ [noise] ABC-123"
)
assert cleaned_example == "เอ่อ อืม ได้ค่ะ ครับ ABC123", cleaned_example

predictions = load_checkpoint(CHECKPOINT_PATH, sample_names)
propagated = propagate_available_originals(predictions)
inference_queue = build_inference_queue(predictions)

print("groups:", len(groups))
print("resumed predictions:", len(predictions))
print("filled/updated from resumed originals:", propagated)
print("model files still needed:", len(inference_queue))
if USE_ORIGINAL_FOR_VARIANTS:
    print("grouping optimization: enabled")


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is strongly recommended for Typhoon Whisper Large v3. "
        "Enable a GPU runtime or switch MODEL_ID to the Turbo model."
    )

device = 0
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32


def load_model_and_processor(target_device, target_dtype):
    model_load_kwargs = dict(
        torch_dtype=target_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )

    # SDPA is available in modern hosted PyTorch images. Fall back cleanly for an
    # older image rather than making model loading brittle.
    try:
        loaded_model = AutoModelForSpeechSeq2Seq.from_pretrained(
            MODEL_ID,
            attn_implementation="sdpa",
            **model_load_kwargs,
        )
    except (TypeError, ValueError) as error:
        print("SDPA load fallback:", error)
        loaded_model = AutoModelForSpeechSeq2Seq.from_pretrained(
            MODEL_ID,
            **model_load_kwargs,
        )

    loaded_model.to(target_device)
    loaded_model.eval()
    loaded_processor = AutoProcessor.from_pretrained(MODEL_ID)
    return loaded_model, loaded_processor


def build_asr_pipeline(target_device, target_dtype, chunk_length_s):
    loaded_model, loaded_processor = load_model_and_processor(target_device, target_dtype)
    loaded_max_target_positions = int(
        getattr(loaded_model.config, "max_target_positions", 448)
    )
    if MAX_NEW_TOKENS >= loaded_max_target_positions:
        raise ValueError(
            f"MAX_NEW_TOKENS={MAX_NEW_TOKENS} must be below the model's "
            f"max_target_positions={loaded_max_target_positions}."
        )

    loaded_asr = pipeline(
        task="automatic-speech-recognition",
        model=loaded_model,
        tokenizer=loaded_processor.tokenizer,
        feature_extractor=loaded_processor.feature_extractor,
        chunk_length_s=chunk_length_s,
        stride_length_s=STRIDE_LENGTH_S,
        max_new_tokens=MAX_NEW_TOKENS,
        batch_size=BATCH_SIZE,
        return_timestamps=True,
        torch_dtype=target_dtype,
        device=0 if str(target_device).startswith("cuda") else -1,
    )
    return loaded_model, loaded_processor, loaded_asr


current_device = "cuda:0" if torch.cuda.is_available() else "cpu"
current_chunk_length_s = CHUNK_LENGTH_S
try:
    model, processor, asr = build_asr_pipeline(
        current_device,
        torch_dtype,
        current_chunk_length_s,
    )
except (RuntimeError, torch.cuda.OutOfMemoryError) as error:
    if not (ALLOW_CPU_FALLBACK and current_device.startswith("cuda") and is_cuda_oom(error)):
        raise
    print("CUDA OOM while loading model: rebuilding pipeline on CPU")
    gc.collect()
    torch.cuda.empty_cache()
    current_device = "cpu"
    torch_dtype = torch.float32
    model, processor, asr = build_asr_pipeline(
        current_device,
        torch_dtype,
        current_chunk_length_s,
    )

generate_kwargs = {
    "language": "thai",
    "task": "transcribe",
}
print("Loaded:", MODEL_ID)


def rebuild_pipeline_for_oom():
    global model, processor, asr, current_device, current_chunk_length_s, torch_dtype

    if current_device.startswith("cuda") and current_chunk_length_s > MIN_CHUNK_LENGTH_S:
        next_chunk_length = max(MIN_CHUNK_LENGTH_S, current_chunk_length_s // 2)
        if next_chunk_length < current_chunk_length_s:
            print(
                "CUDA OOM: rebuilding pipeline with "
                f"chunk_length_s={next_chunk_length}"
            )
            del model, processor, asr
            gc.collect()
            torch.cuda.empty_cache()
            current_chunk_length_s = next_chunk_length
            model, processor, asr = build_asr_pipeline(
                current_device,
                torch_dtype,
                current_chunk_length_s,
            )
            return True

    if ALLOW_CPU_FALLBACK and current_device.startswith("cuda"):
        print("CUDA OOM: rebuilding pipeline on CPU")
        del model, processor, asr
        gc.collect()
        torch.cuda.empty_cache()
        current_device = "cpu"
        torch_dtype = torch.float32
        model, processor, asr = build_asr_pipeline(
            current_device,
            torch_dtype,
            current_chunk_length_s,
        )
        return True

    return False


In [ ]:
def load_wav_for_pipeline(path):
    '''
    Read WAV directly instead of passing a filename to Transformers. Passing
    filenames requires an external ffmpeg executable, which is often absent on
    local Windows installations.
    '''
    audio, sampling_rate = sf.read(
        str(path),
        dtype="float32",
        always_2d=False,
    )
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    target_rate = int(processor.feature_extractor.sampling_rate)
    if sampling_rate != target_rate:
        audio = librosa.resample(
            audio,
            orig_sr=sampling_rate,
            target_sr=target_rate,
        )
        sampling_rate = target_rate
    audio = np.ascontiguousarray(audio, dtype=np.float32)
    return {
        "raw": audio,
        "sampling_rate": int(sampling_rate),
    }


def transcribe_adaptive(paths, batch_size):
    '''
    Infer a list of paths. On CUDA OOM, first reduce the pipeline batch size,
    then split the outer list. A one-file, batch-size-1 OOM is surfaced with a
    useful error instead of silently producing an empty transcript.
    '''
    audio_inputs = [load_wav_for_pipeline(path) for path in paths]
    try:
        outputs = asr(
            audio_inputs,
            batch_size=batch_size,
            generate_kwargs=generate_kwargs,
        )
        if isinstance(outputs, dict):
            outputs = [outputs]
        return [clean_thai_transcript(output.get("text", "")) for output in outputs]
    except RuntimeError as error:
        if not is_cuda_oom(error):
            raise
        gc.collect()
        torch.cuda.empty_cache()
        if batch_size > 1:
            smaller = max(1, batch_size // 2)
            print(f"CUDA OOM: retrying with pipeline batch_size={smaller}")
            return transcribe_adaptive(paths, smaller)
        if rebuild_pipeline_for_oom():
            return transcribe_adaptive(paths, 1)
        if len(paths) > 1:
            midpoint = len(paths) // 2
            return (
                transcribe_adaptive(paths[:midpoint], 1)
                + transcribe_adaptive(paths[midpoint:], 1)
            )
        raise RuntimeError(
            "CUDA OOM on one file at batch_size=1. Switch to the Turbo model "
            "or reduce CHUNK_LENGTH_S (for example, to 20)."
        ) from error


queue = inference_queue
if MAX_MODEL_FILES is not None:
    queue = queue[:MAX_MODEL_FILES]
    print(f"Smoke-test mode: limited to {len(queue)} model files.")

inferred_since_save = 0
total_new_model_predictions = 0
started_at = time.perf_counter()


def format_seconds(seconds):
    seconds = max(0, int(round(seconds)))
    minutes, remaining_seconds = divmod(seconds, 60)
    hours, remaining_minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h{remaining_minutes:02d}m{remaining_seconds:02d}s"
    if minutes:
        return f"{minutes}m{remaining_seconds:02d}s"
    return f"{remaining_seconds}s"

try:
    total_batches = max(1, (len(queue) + BATCH_SIZE - 1) // BATCH_SIZE)
    for batch_index, start in enumerate(range(0, len(queue), BATCH_SIZE), start=1):
        names = queue[start : start + BATCH_SIZE]
        paths = [audio_index[name] for name in names]
        texts = transcribe_adaptive(paths, BATCH_SIZE)
        if len(texts) != len(names):
            raise RuntimeError(
                f"ASR returned {len(texts)} results for {len(names)} inputs."
            )

        for name, text in zip(names, texts):
            predictions[name] = text
            total_new_model_predictions += 1
            inferred_since_save += 1

            if USE_ORIGINAL_FOR_VARIANTS and name == original_file_name(name):
                for member in groups[name]:
                    predictions[member] = text

        elapsed = time.perf_counter() - started_at
        done = min(start + len(names), len(queue))
        rate = done / elapsed if elapsed > 0 else 0.0
        remaining = len(queue) - done
        eta = remaining / rate if rate > 0 else 0.0
        print(
            f"Progress: {done}/{len(queue)} model files "
            f"({done / len(queue) * 100:.1f}%), "
            f"batch {batch_index}/{total_batches}, "
            f"elapsed {format_seconds(elapsed)}, ETA {format_seconds(eta)}"
        )

        if inferred_since_save >= CHECKPOINT_EVERY:
            save_checkpoint(CHECKPOINT_PATH, sample_names, predictions)
            print(
                f"Checkpoint: {len(predictions)}/{len(sample_names)} rows "
                f"({total_new_model_predictions}/{len(queue)} model files inferred)"
            )
            inferred_since_save = 0

except BaseException:
    save_checkpoint(CHECKPOINT_PATH, sample_names, predictions)
    print(f"Saved emergency checkpoint with {len(predictions)} rows.")
    raise
finally:
    save_checkpoint(CHECKPOINT_PATH, sample_names, predictions)

elapsed_total = time.perf_counter() - started_at
print("Inference complete.")
print("New model predictions:", total_new_model_predictions)
print("Predicted sample rows after propagation:", len(predictions))
print("Elapsed:", format_seconds(elapsed_total))
print("Checkpoint saved:", CHECKPOINT_PATH)


In [ ]:
# Re-apply grouping in case the checkpoint came from a previous partial run,
# then construct the final frame strictly from the sample's order.
propagate_available_originals(predictions)
missing_predictions = [name for name in sample_names if name not in predictions]
if missing_predictions:
    raise RuntimeError(
        f"Cannot export an incomplete submission: {len(missing_predictions)} "
        f"predictions are missing. Examples: {missing_predictions[:5]}. "
        "Set MAX_MODEL_FILES=None and run the inference cell again."
    )

submission = pd.DataFrame({
    "file_name": sample_names,
    "text": [clean_thai_transcript(predictions[name]) for name in sample_names],
})

# Competition-critical validation.
assert submission.columns.tolist() == ["file_name", "text"]
assert len(submission) == len(sample)
assert submission["file_name"].tolist() == sample["file_name"].tolist()
assert not submission["file_name"].isna().any()
assert not submission["text"].isna().any()
assert not submission["file_name"].duplicated().any()

submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

# Read it back to catch serialization or path mistakes before submission.
round_trip = pd.read_csv(SUBMISSION_PATH, keep_default_na=False)
assert round_trip.columns.tolist() == ["file_name", "text"]
assert round_trip["file_name"].tolist() == sample_names
assert len(round_trip) == len(sample_names)

print("submission:", SUBMISSION_PATH)
print("rows:", len(submission))
print("blank transcripts:", int(submission["text"].eq("").sum()))
print("Done: submission.csv is ready.")
try:
    display(submission.head(10))
except NameError:
    print(submission.head(10).to_string(index=False))


In [ ]:
# Optional Colab download. On Kaggle, submission.csv appears in the Output tab.
try:
    from google.colab import files
    files.download(str(SUBMISSION_PATH))
except ImportError:
    print(f"Use this file for Kaggle submission: {SUBMISSION_PATH}")
